In [1]:
import numpy as np
import tensorflow as tf

2026-03-23 00:38:20.764541: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-23 00:38:20.765037: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-23 00:38:20.767794: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-23 00:38:20.775687: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-23 00:38:20.790131: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

Load the data

In [2]:
npz = np.load('Audiobooks_data_train.npz')
train_inputs = npz['inputs'].astype(float)
train_targets = npz['outputs'].astype(int)

npz = np.load('Audiobooks_data_validation.npz')
validation_inputs = npz['inputs'].astype(float)
validation_targets = npz['outputs'].astype(int)

npz = np.load('Audiobooks_data_test.npz')
test_inputs = npz['inputs'].astype(float)
test_targets = npz['outputs'].astype(int)




Model

In [3]:
input_size = train_inputs.shape[1]
output_size = 2
hidden_layer_size = 50

model = tf.keras.Sequential([
#        tf.keras.layers.Flatten(input_size=(X,Y,1)),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
        tf.keras.layers.Dense(output_size, activation='softmax')
])
#model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['accuracy'])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

batch_size = 100
epochs = 150
early_stopping = tf.keras.callbacks.EarlyStopping(patience=2)

model.fit(train_inputs,
         train_targets,
         batch_size = batch_size,
         epochs = epochs,
         callbacks=[early_stopping],
         validation_data=(validation_inputs, validation_targets),
         verbose=2)

Epoch 1/150
36/36 - 1s - 23ms/step - accuracy: 0.6787 - loss: 0.5961 - val_accuracy: 0.7226 - val_loss: 0.5237
Epoch 2/150
36/36 - 0s - 2ms/step - accuracy: 0.7675 - loss: 0.4695 - val_accuracy: 0.7763 - val_loss: 0.4485
Epoch 3/150
36/36 - 0s - 2ms/step - accuracy: 0.7918 - loss: 0.4127 - val_accuracy: 0.7606 - val_loss: 0.4207
Epoch 4/150
36/36 - 0s - 2ms/step - accuracy: 0.8016 - loss: 0.3836 - val_accuracy: 0.7919 - val_loss: 0.3949
Epoch 5/150
36/36 - 0s - 2ms/step - accuracy: 0.8061 - loss: 0.3702 - val_accuracy: 0.7875 - val_loss: 0.3855
Epoch 6/150
36/36 - 0s - 2ms/step - accuracy: 0.8120 - loss: 0.3580 - val_accuracy: 0.7919 - val_loss: 0.3789
Epoch 7/150
36/36 - 0s - 2ms/step - accuracy: 0.8128 - loss: 0.3512 - val_accuracy: 0.8098 - val_loss: 0.3712
Epoch 8/150
36/36 - 0s - 2ms/step - accuracy: 0.8064 - loss: 0.3502 - val_accuracy: 0.7673 - val_loss: 0.3842
Epoch 9/150
36/36 - 0s - 2ms/step - accuracy: 0.8145 - loss: 0.3455 - val_accuracy: 0.7897 - val_loss: 0.3646
Epoch 10/

In [4]:
test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)


14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8013 - loss: 0.3672 


In [5]:
import joblib
joblib.dump(model, "model.pkl")

['model.pkl']

In [6]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

# y_pred_train = (model.predict(train_inputs)[:,1] > 0.6).astype(int)
# y_pred_val = (model.predict(validation_inputs)[:,1] > 0.6).astype(int)
# y_pred_test = (model.predict(test_inputs) > 0.6)[:,1].astype(int)

y_pred_train = np.argmax(model.predict(train_inputs), axis=1)
y_pred_val = np.argmax(model.predict(validation_inputs), axis=1)
y_pred_test = np.argmax(model.predict(test_inputs), axis=1)

#accuracy = model.score(test_inputs, test_targets)
cm = confusion_matrix(test_targets, y_pred_test)
accuracy = accuracy_score(test_targets, y_pred_test)
precision = precision_score(test_targets, y_pred_test)
recall = recall_score(test_targets, y_pred_test)
f1 = f1_score(test_targets, y_pred_test)

print("Confusion Matrix:\n", cm)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 728us/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Confusion Matrix:
 [[179  40]
 [ 49 180]]
Accuracy: 0.8013392857142857
Precision: 0.8181818181818182
Recall: 0.7860262008733624
F1 Score: 0.8017817371937639


In [7]:
np.savez('Audiobooks_data_train_pred', inputs=train_inputs, outputs=y_pred_train)
np.savez('Audiobooks_data_validation_pred', inputs=validation_inputs, outputs=y_pred_val)
np.savez('Audiobooks_data_test_pred', inputs=test_inputs, outputs=y_pred_test)

In [8]:
# import pandas as pd

# df = pd.DataFrame({
#     'y_true': test_targets,
#     'y_score': y_pred_test
# })

# df = df.sort_values(by='y_score', ascending=False)
# df = df.reset_index(drop=True)

In [9]:

# baseline = df['y_true'].mean() #out of all the users how many actually bought

# def precision_at_k(df, k):
#     cutoff = int(len(df) * k)
#     top_k = df.iloc[:cutoff]
#     return top_k['y_true'].mean()

# k=0.1
# p10 = precision_at_k(df, k)
# p20 = precision_at_k(df, 0.2)
# print("Precision@10%:", precision_at_k(df, p10))
# print("Precision@20%:", precision_at_k(df, p20))

# print(f"If baseline conversion = {(baseline*100).round(2)}% then Precision@10% = {(p10*100).round(2)}%")
# print(f"If I target top {k*100}% users, {(p10*100).round(2)}% may actually convert.")

Precision@10%: 0.6101190476190477
Precision@20%: 0.59375
If baseline conversion = 51.12% then Precision@10% = 75.0%
If I target top 10.0% users, 75.0% may actually convert.
